In [279]:
from pydantic import BaseModel, Field
from typing import List, Optional
from google import genai
import os
from dotenv import load_dotenv

# Schemas

### Exam Paper

In [280]:
class SubjectiveQuestion(BaseModel):
    question_id: str = Field(description="""
        Unique id for exam in this format: 
        Q(n) where n is question number
        Q(n(m)) where n is outer question number while m is inner question number
        and so on...
        """)
    marks: float = Field(description="Marks of the question or sum of marks of inner questions")
    question_statement: str = Field(description="Statement or description the of the question")
    questions: List[SubjectiveQuestion] | None = Field(default=None, description="Sub questions in question: Q(n(m)) case (if any)")

In [281]:
# class ObjectiveOption(BaseModel):
#     option: str = Field(description="Option identifier like: A, B, C... or i, ii, iii...")
#     value: str = Field(description="Option's value")

In [282]:
class ObjectiveQuestion(BaseModel):
    question_id: str = Field(description="""
        Unique id for exam in this format: 
        Q(n) where n is question number
        Q(n(m)) where n is outer question number while m is inner question number
        and so on...
        """)
    marks: float = Field(description="Mark of the MCQ")
    mcq_statement: str = Field(description="Statement MCQ and its options")

In [283]:
class ExamPaper(BaseModel):
    subject : str = Field(description="Name of the subject")
    course_id: str | None = Field(default=None, description="ID of paper (only if available)")
    date: str | None = Field(default=None, description="Date of exam (only if available)")
    total_marks: float = Field(description="Total marks of all questions in the exam")
    objective: List[ObjectiveQuestion] | None = Field(default=None, description="Collection of Objective MCQs")
    subjective: List[SubjectiveQuestion] | None = Field(default=None, description="Collection of questions and sub questions")

### Answer Paper

In [284]:
class ObjectiveAnswer(BaseModel):
    question_id: str = Field(description="""
        Unique id for exam in this format: 
        Q(n) where n is question number
        Q(n(m)) where n is outer question number while m is inner question number (when MCQs portion is part of super question)
        and so on...
        """)
    chosen_option: str = Field(description="The identifier of option chosen for MCQ")

In [285]:
class SubjectiveAnswer(BaseModel):
    question_id: str = Field(description="""
        Unique id for exam in this format: 
        Q(n) where n is question number
        Q(n(m)) where n is outer question number while m is inner question number
        and so on...
    """)
    answer_statement: str | None = Field(default=None, description="Statement or text of the answer (A question will either have answer statement or sub answers)")
    answers: List[SubjectiveAnswer] | None = Field(default=None, description="Sub answers in question: Q(n(m)) case (A question will either have answer statement or sub answers)")


In [286]:
class AnswerPaper(BaseModel):
    subject : str = Field(description="Name of the subject")
    roll_no: str = Field(description="Roll number of student")
    name: str | None = Field(default=None, description="Name of student (if available)")
    objective: List[ObjectiveAnswer] | None = Field(default=None, description="Collectection of MCQ chose options")
    subjective : List[SubjectiveAnswer] = Field(default=None, description="Collection of answers and sub answers")
    

# Formatter

In [287]:
class ExamFormatter:

    def __init__(self, gemini_api_key: str):
        self.client = genai.Client(
            api_key=gemini_api_key    
        )

    def format_exam_paper(self, exam_paper: str, model: str = "gemini-3.5-flash-lite"):
        response = self.client.interactions.create(
            model=model,
            input=f"""
                Format this exam [a[er]] according to given schema. 
                You will encounter questions, sub questions and marks written in front of each question.
                Exam paper: {exam_paper}
            """,
            response_format={
                "type": "text",
                "mime_type": "application/json",
                "schema": ExamPaper.model_json_schema()
            },
        )
        formatted_exam_paper = ExamPaper.model_validate_json(response.output_text)
        return formatted_exam_paper

    def format_answer_paper(self, answer_paper: str, model: str = "gemini-3.5-flash-lite"):
        response = self.client.interactions.create(
            model=model,
            input=f"""
                Format this answer paper according to given schema. 
                You will encounter answers and subanswers. You have to extract question id of sub questions yourself.
                Exam paper: {answer_paper}
            """,
            response_format={
                "type": "text",
                "mime_type": "application/json",
                "schema": AnswerPaper.model_json_schema()
            },
        )
        formatted_answer_paper = AnswerPaper.model_validate_json(response.output_text)
        return formatted_answer_paper

### Testing

In [288]:
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

In [289]:
exam_formatter = ExamFormatter(GEMINI_API_KEY)

In [290]:
exam_text = """
# Data Structures and Algorithms

### Mid-Term Examination

**Time:** 45 Minutes
**Total Marks:** 20

---

**Q1**. Choose the correct option

**Q1.** Which data structure follows the **LIFO** principle? **[1]**

A. Queue
B. Stack
C. Linked List
D. Tree

**Q2.** Which traversal of a Binary Search Tree produces the values in sorted order? **[1]**

A. Preorder
B. Postorder
C. Inorder
D. Level Order

**Q3.** What is the worst-case time complexity of binary search? **[1]**

A. O(1)
B. O(log n)
C. O(n)
D. O(n²)

**Q4.** Which of the following is **not** a linear data structure? **[1]**

A. Array
B. Stack
C. Queue
D. Binary Tree

---


**Q2.** Answer the following:

**(a)** Define a stack and explain the LIFO principle. **[2]**

**(b)** Give two real-world applications of stacks. **[2]**

---

**Q3.** Consider the following sequence of queue operations:

`ENQUEUE(10), ENQUEUE(20), DEQUEUE(), ENQUEUE(30), DEQUEUE()`

**(a)** What value is returned by the first `DEQUEUE()` operation? **[1]**

**(b)** What value is returned by the second `DEQUEUE()` operation? **[1]**

**(c)** What element(s) remain in the queue after all operations? **[2]**

---

**Q4.** Consider the following Binary Search Tree:

```text
        50
       /  \
     30    70
    / \    / \
   20 40  60 80
```

**(a)** Write the inorder traversal of the tree. **[2]**

**(b)** Explain why the inorder traversal of a Binary Search Tree produces sorted values. **[2]**

---
"""

<>:70: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
<>:70: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
C:\Users\User\AppData\Local\Temp\ipykernel_18196\2909823528.py:70: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
  / \    / \


In [291]:
formatted_exam = exam_formatter.format_exam_paper(exam_text)

In [292]:
formatted_exam

ExamPaper(subject='Data Structures and Algorithms', course_id=None, date=None, total_marks=20.0, objective=[ObjectiveQuestion(question_id='Q1(1)', marks=1.0, mcq_statement='Which data structure follows the LIFO principle?\nA. Queue\nB. Stack\nC. Linked List\nD. Tree'), ObjectiveQuestion(question_id='Q1(2)', marks=1.0, mcq_statement='Which traversal of a Binary Search Tree produces the values in sorted order?\nA. Preorder\nB. Postorder\nC. Inorder\nD. Level Order'), ObjectiveQuestion(question_id='Q1(3)', marks=1.0, mcq_statement='What is the worst-case time complexity of binary search?\nA. O(1)\nB. O(log n)\nC. O(n)\nD. O(n²)'), ObjectiveQuestion(question_id='Q1(4)', marks=1.0, mcq_statement='Which of the following is not a linear data structure?\nA. Array\nB. Stack\nC. Queue\nD. Binary Tree')], subjective=[SubjectiveQuestion(question_id='Q2', marks=4.0, question_statement='Answer the following:', questions=[SubjectiveQuestion(id='Q2(1)', marks=2.0, question_statement='Define a stack an

In [293]:
print(formatted_exam.subject)
print(formatted_exam.total_marks)
print(formatted_exam.date)

if formatted_exam.objective is not None:
    for question in formatted_exam.objective:
        print(question)
else:
    print("No objective")

if formatted_exam.subjective is not None:
    for question in formatted_exam.subjective:
        print(question)
        if question.questions is not None:
            for inner_answer in question.questions:
                print(inner_answer)
else:
    print("No subjective")

Data Structures and Algorithms
20.0
None
question_id='Q1(1)' marks=1.0 mcq_statement='Which data structure follows the LIFO principle?\nA. Queue\nB. Stack\nC. Linked List\nD. Tree'
question_id='Q1(2)' marks=1.0 mcq_statement='Which traversal of a Binary Search Tree produces the values in sorted order?\nA. Preorder\nB. Postorder\nC. Inorder\nD. Level Order'
question_id='Q1(3)' marks=1.0 mcq_statement='What is the worst-case time complexity of binary search?\nA. O(1)\nB. O(log n)\nC. O(n)\nD. O(n²)'
question_id='Q1(4)' marks=1.0 mcq_statement='Which of the following is not a linear data structure?\nA. Array\nB. Stack\nC. Queue\nD. Binary Tree'
question_id='Q2' marks=4.0 question_statement='Answer the following:' questions=[SubjectiveQuestion(id='Q2(1)', marks=2.0, question_statement='Define a stack and explain the LIFO principle.', questions=None), SubjectiveQuestion(id='Q2(2)', marks=2.0, question_statement='Give two real-world applications of stacks.', questions=None)]
id='Q2(1)' marks

In [294]:
answer_paper = """
# Data Structures and Algorithms

### Mid-Term Examination

**Roll No:** 1126
**Name:** Ali Khan

---

**Q1** Choose the correct options 

**Q1.** B — Stack

**Q2.** C

**Q3.** B) O(log n)

**Q4.** D. Binary Tree

---

## Part II

### Q2

**(a)** A stack is a linear data structure in which insertion and deletion are performed from the same end called the top. It follows the LIFO principle, meaning the last element inserted is the first one removed.

**(b)** Two real-world applications of stacks are:

1. Undo/redo operations in software.
2. Browser back button / browser history.

---

### Q3

**(a)** The first DEQUEUE() removes **10**.

**(b)** The second DEQUEUE() removes **20**.

**(c)** After all operations, **30** remains in the queue.

---

### Q4

**(a)** Inorder traversal:

`20, 30, 40, 50, 60, 70, 80`

**(b)** In a Binary Search Tree, every value in the left subtree is smaller than the node and every value in the right subtree is greater. Therefore, visiting the left subtree, then the root, and then the right subtree produces the values in ascending order.

"""

In [295]:
formatted_answer_paper = exam_formatter.format_answer_paper(answer_paper)

In [296]:
formatted_answer_paper

AnswerPaper(subject='Data Structures and Algorithms', roll_no='1126', name='Ali Khan', objective=[ObjectiveAnswer(question_id='Q1(1)', chosen_option='B'), ObjectiveAnswer(question_id='Q1(2)', chosen_option='C'), ObjectiveAnswer(question_id='Q1(3)', chosen_option='B'), ObjectiveAnswer(question_id='Q1(4)', chosen_option='D')], subjective=[SubjectiveAnswer(question_id='Q2', answer_statement=None, answers=[SubjectiveAnswer(question_id='Q2(a)', answer_statement='A stack is a linear data structure in which insertion and deletion are performed from the same end called the top. It follows the LIFO principle, meaning the last element inserted is the first one removed.', answers=None), SubjectiveAnswer(question_id='Q2(b)', answer_statement='Two real-world applications of stacks are:\n\n1. Undo/redo operations in software.\n2. Browser back button / browser history.', answers=None)]), SubjectiveAnswer(question_id='Q3', answer_statement=None, answers=[SubjectiveAnswer(question_id='Q3(a)', answer_st

In [297]:
print(formatted_answer_paper.subject)
print(formatted_answer_paper.roll_no)
print(formatted_answer_paper.name)

if formatted_answer_paper.objective:
    for answer in formatted_answer_paper.objective:
        print(answer)

if formatted_answer_paper.subjective:
    for answer in formatted_answer_paper.subjective:
        print(answer)
        if answer.answers is not None:
            for inner_answer in answer.answers:
                print(inner_answer)

Data Structures and Algorithms
1126
Ali Khan
question_id='Q1(1)' chosen_option='B'
question_id='Q1(2)' chosen_option='C'
question_id='Q1(3)' chosen_option='B'
question_id='Q1(4)' chosen_option='D'
question_id='Q2' answer_statement=None answers=[SubjectiveAnswer(question_id='Q2(a)', answer_statement='A stack is a linear data structure in which insertion and deletion are performed from the same end called the top. It follows the LIFO principle, meaning the last element inserted is the first one removed.', answers=None), SubjectiveAnswer(question_id='Q2(b)', answer_statement='Two real-world applications of stacks are:\n\n1. Undo/redo operations in software.\n2. Browser back button / browser history.', answers=None)]
question_id='Q2(a)' answer_statement='A stack is a linear data structure in which insertion and deletion are performed from the same end called the top. It follows the LIFO principle, meaning the last element inserted is the first one removed.' answers=None
question_id='Q2(b)'